# Model Context Protocol (MCP) — Building It from Scratch

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/29_mcp_from_scratch.ipynb)

## What This Notebook Teaches You

Every LLM tool integration today is custom-built. OpenAI function calling, Anthropic tool use, LangChain tools — they all define tools differently. **Model Context Protocol (MCP)** is Anthropic's answer to this fragmentation: a universal, open protocol for connecting LLMs to tools and data sources.

**Research foundation**:
- Anthropic, *"Model Context Protocol Specification"* (2024) — an open standard for LLM-tool communication
- The core insight: separate the **protocol** (how to communicate) from the **implementation** (what tools do), just like HTTP separated web communication from web applications

By the end of this notebook, you will:

1. **Understand the MCP architecture** — hosts, clients, servers, and transport layers
2. **Build a complete MCP server** that registers and exposes tools with JSON Schema definitions
3. **Build an MCP client** that discovers and invokes tools via a structured protocol
4. **Implement JSON-RPC-style messaging** with requests, responses, and error handling
5. **Build an MCP-aware agent** that dynamically discovers tools from servers
6. **Connect an agent to multiple MCP servers** for cross-domain tool access
7. **Compare hardcoded vs. protocol-based tool integration**

---

> **Prerequisites**: Notebooks 01–06 (agent fundamentals, tool use, ReAct, planning).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~55–70 minutes.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **What This Notebook Teaches You**
- Understand **: Environment Setup**
- Understand **What Is MCP?**
- Understand **MCP Architecture Overview**
- Understand **Communication Protocol — JSON-RPC Messages**


## Part 0: Environment Setup

Same Qwen3-8B setup. The model will serve as the reasoning engine inside our MCP-aware agent, deciding which tools to call based on dynamically discovered capabilities.

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. What Is MCP?

### 1.1 — The Problem: Tool Integration Fragmentation

Today, every framework defines tools differently:

```
OpenAI:      {"type": "function", "function": {"name": "...", "parameters": {...}}}
LangChain:   @tool decorator with docstring parsing
AutoGen:     register_for_llm() + register_for_execution()
Custom:      Whatever JSON format you invented last Tuesday
```

This means:
- **Tools aren't portable** — a tool built for LangChain can't be used in AutoGen
- **Servers can't be shared** — every team builds their own tool wrappers
- **Discovery is manual** — you hardcode which tools an agent has access to

### 1.2 — The Solution: A Universal Protocol

**Model Context Protocol (MCP)** standardizes the interface between LLM applications and tool providers, the same way HTTP standardized the interface between web browsers and web servers.

```
Before HTTP:     Every app had its own network protocol
After HTTP:      Any browser talks to any server

Before MCP:      Every LLM app has its own tool format
After MCP:       Any LLM client talks to any tool server
```

### 1.3 — Key Design Principles

1. **Server-centric tools**: Tools live on servers, not inside the LLM application
2. **Dynamic discovery**: Clients ask servers what tools are available at runtime
3. **Schema-driven**: Every tool describes its inputs using JSON Schema
4. **Transport-agnostic**: Works over stdio, HTTP+SSE, or any transport layer
5. **Capability negotiation**: Client and server agree on what features they support

## 2. MCP Architecture Overview

### 2.1 — The Three Roles

```
┌─────────────────────────────────────────────────────────────┐
│                        HOST APPLICATION                      │
│  (e.g., Claude Desktop, IDE plugin, your custom app)        │
│                                                              │
│   ┌──────────────┐   ┌──────────────┐   ┌──────────────┐   │
│   │  MCP Client  │   │  MCP Client  │   │  MCP Client  │   │
│   │  (handler)   │   │  (handler)   │   │  (handler)   │   │
│   └──────┬───────┘   └──────┬───────┘   └──────┬───────┘   │
│          │                  │                   │            │
└──────────┼──────────────────┼───────────────────┼────────────┘
           │ protocol         │ protocol          │ protocol
           ▼                  ▼                   ▼
    ┌──────────────┐   ┌──────────────┐   ┌──────────────┐
    │  MCP Server  │   │  MCP Server  │   │  MCP Server  │
    │  (calculator)│   │  (database)  │   │  (web search) │
    └──────────────┘   └──────────────┘   └──────────────┘
```

- **Host**: The application the user interacts with (IDE, chatbot, etc.)
- **Client**: Protocol handler that maintains a 1:1 connection with a server
- **Server**: Exposes tools, resources, and prompts via the MCP protocol

### 2.2 — What Flows Over the Protocol

| Direction | Message Type | Example |
|-----------|-------------|---------|
| Client → Server | `tools/list` | "What tools do you have?" |
| Client → Server | `tools/call` | "Run the `add` tool with {a: 3, b: 5}" |
| Server → Client | Response | "{result: 8}" or "{error: ...}" |
| Both | Notifications | "Tool list changed" / "Progress update" |

## 3. Communication Protocol — JSON-RPC Messages

MCP uses **JSON-RPC 2.0** as its message format. This is a well-established standard for remote procedure calls that gives us three message types:

1. **Request** — client asks the server to do something (expects a response)
2. **Response** — server replies with a result or error
3. **Notification** — one-way message, no response expected

This is the same pattern used by Language Server Protocol (LSP) in IDEs — proven at scale.

In [ ]:
# ============================================================
# MCP Message Format (JSON-RPC 2.0 inspired)
# ============================================================

import uuid

@dataclass
class MCPRequest:
    """A request from client to server. Expects a response."""
    method: str                    # e.g., "tools/list", "tools/call"
    params: Dict[str, Any]         # method-specific parameters
    id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])

    def to_dict(self) -> dict:
        return {
            "jsonrpc": "2.0",
            "method": self.method,
            "params": self.params,
            "id": self.id
        }

@dataclass
class MCPResponse:
    """A successful response from server to client."""
    result: Any                    # the actual result data
    id: str = ""                   # matches the request id

    def to_dict(self) -> dict:
        return {
            "jsonrpc": "2.0",
            "result": self.result,
            "id": self.id
        }

@dataclass
class MCPError:
    """An error response from server to client."""
    code: int                      # error code (like HTTP status codes)
    message: str                   # human-readable error description
    id: str = ""                   # matches the request id
    data: Any = None               # optional additional error info

    def to_dict(self) -> dict:
        result = {
            "jsonrpc": "2.0",
            "error": {"code": self.code, "message": self.message},
            "id": self.id
        }
        if self.data:
            result["error"]["data"] = self.data
        return result

@dataclass
class MCPNotification:
    """A one-way message (no response expected)."""
    method: str
    params: Dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> dict:
        return {
            "jsonrpc": "2.0",
            "method": self.method,
            "params": self.params
        }

# Standard MCP error codes (inspired by JSON-RPC and HTTP)
class ErrorCode:
    PARSE_ERROR = -32700       # invalid JSON
    INVALID_REQUEST = -32600   # malformed request
    METHOD_NOT_FOUND = -32601  # unknown method
    INVALID_PARAMS = -32602    # bad parameters
    INTERNAL_ERROR = -32603    # server-side failure
    TOOL_NOT_FOUND = -32001    # custom: requested tool doesn't exist
    TOOL_EXECUTION_ERROR = -32002  # custom: tool raised an exception

# Demonstrate the message format
req = MCPRequest(method="tools/list", params={})
print("Request:", json.dumps(req.to_dict(), indent=2))

resp = MCPResponse(result={"tools": [{"name": "add"}]}, id=req.id)
print("\nResponse:", json.dumps(resp.to_dict(), indent=2))

err = MCPError(code=ErrorCode.TOOL_NOT_FOUND, message="Tool 'foo' not found", id="abc123")
print("\nError:", json.dumps(err.to_dict(), indent=2))

## 4. Building the MCP Tool Schema

In MCP, every tool is described by a **schema** that tells clients:
- **What** the tool does (name + description)
- **What inputs** it accepts (JSON Schema for parameters)

This is the key to dynamic discovery — a client can read these schemas at runtime and know exactly how to call any tool, without hardcoded knowledge.

The schema format follows JSON Schema (RFC draft-bhutton-json-schema-01), the same standard used by OpenAI function calling.

In [ ]:
# ============================================================
# MCP Tool Definition Schema
# ============================================================

@dataclass
class ToolSchema:
    """Describes a single tool that an MCP server exposes."""
    name: str                      # unique tool identifier
    description: str               # human-readable description (shown to LLM)
    input_schema: Dict[str, Any]   # JSON Schema for the tool's parameters

    def to_dict(self) -> dict:
        """Convert to MCP-compatible tool definition."""
        return {
            "name": self.name,
            "description": self.description,
            "inputSchema": self.input_schema
        }

# Example: define a calculator "add" tool
add_schema = ToolSchema(
    name="add",
    description="Add two numbers together and return the sum.",
    input_schema={
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"}
        },
        "required": ["a", "b"]
    }
)

print("Tool schema:")
print(json.dumps(add_schema.to_dict(), indent=2))
print()
print("This tells any MCP client exactly how to call this tool —")
print("no documentation reading, no guessing, no hardcoding.")

## 5. Building the MCP Server

The server is the tool provider. It:
1. **Registers** tools with their schemas and handler functions
2. **Responds** to `tools/list` requests with available tool schemas
3. **Executes** `tools/call` requests by running the appropriate handler
4. **Returns** results or errors in the standard MCP format

Think of it like a web server: it doesn't initiate communication, it responds to requests.

In [ ]:
# ============================================================
# MCP Server — registers tools and handles requests
# ============================================================

class MCPServer:
    """A simplified MCP server that registers and exposes tools."""

    def __init__(self, name: str, version: str = "1.0"):
        self.name = name
        self.version = version
        self.tools: Dict[str, ToolSchema] = {}
        self.handlers: Dict[str, Callable] = {}
        self.capabilities = {
            "tools": {"listChanged": True}  # we support tool listing
        }
        self._log: List[str] = []

    def register_tool(
        self,
        name: str,
        description: str,
        input_schema: Dict[str, Any],
        handler: Callable
    ) -> None:
        """Register a new tool with its schema and handler function."""
        schema = ToolSchema(name=name, description=description, input_schema=input_schema)
        self.tools[name] = schema
        self.handlers[name] = handler
        self._log.append(f"Registered tool: {name}")

    def handle_request(self, request: MCPRequest) -> Union[MCPResponse, MCPError]:
        """Route an incoming request to the appropriate handler."""
        self._log.append(f"Received: {request.method} (id={request.id})")

        if request.method == "initialize":
            return self._handle_initialize(request)
        elif request.method == "tools/list":
            return self._handle_list_tools(request)
        elif request.method == "tools/call":
            return self._handle_call_tool(request)
        else:
            return MCPError(
                code=ErrorCode.METHOD_NOT_FOUND,
                message=f"Unknown method: {request.method}",
                id=request.id
            )

    def _handle_initialize(self, request: MCPRequest) -> MCPResponse:
        """Handle the initialization handshake."""
        return MCPResponse(
            result={
                "protocolVersion": "2024-11-05",
                "serverInfo": {"name": self.name, "version": self.version},
                "capabilities": self.capabilities
            },
            id=request.id
        )

    def _handle_list_tools(self, request: MCPRequest) -> MCPResponse:
        """Return all registered tool schemas."""
        tools_list = [schema.to_dict() for schema in self.tools.values()]
        return MCPResponse(result={"tools": tools_list}, id=request.id)

    def _handle_call_tool(self, request: MCPRequest) -> Union[MCPResponse, MCPError]:
        """Execute a tool and return its result."""
        tool_name = request.params.get("name", "")
        arguments = request.params.get("arguments", {})

        if tool_name not in self.tools:
            return MCPError(
                code=ErrorCode.TOOL_NOT_FOUND,
                message=f"Tool '{tool_name}' not found. Available: {list(self.tools.keys())}",
                id=request.id
            )

        try:
            result = self.handlers[tool_name](**arguments)
            return MCPResponse(
                result={"content": [{"type": "text", "text": str(result)}]},
                id=request.id
            )
        except TypeError as e:
            return MCPError(
                code=ErrorCode.INVALID_PARAMS,
                message=f"Invalid parameters for '{tool_name}': {e}",
                id=request.id
            )
        except Exception as e:
            return MCPError(
                code=ErrorCode.TOOL_EXECUTION_ERROR,
                message=f"Tool '{tool_name}' failed: {e}",
                id=request.id
            )

    def get_log(self) -> List[str]:
        return list(self._log)

# Quick test
server = MCPServer("test-server")
server.register_tool(
    "greet", "Say hello to someone.",
    {"type": "object", "properties": {"name": {"type": "string"}}, "required": ["name"]},
    lambda name: f"Hello, {name}!"
)

# Test list tools
req = MCPRequest(method="tools/list", params={})
resp = server.handle_request(req)
print("tools/list response:")
print(json.dumps(resp.to_dict(), indent=2))

# Test call tool
req2 = MCPRequest(method="tools/call", params={"name": "greet", "arguments": {"name": "Alice"}})
resp2 = server.handle_request(req2)
print("\ntools/call response:")
print(json.dumps(resp2.to_dict(), indent=2))

## 6. Building the MCP Client

The client is the protocol handler inside the host application. It:
1. **Connects** to a server and performs the initialization handshake
2. **Discovers** available tools by querying the server
3. **Calls** tools by sending properly formatted requests
4. **Handles** responses and errors gracefully

In a real MCP implementation, the client would communicate over stdio or HTTP. In our simplified version, we call the server's `handle_request` method directly — the message format is identical either way.

In [ ]:
# ============================================================
# MCP Client — connects to servers and invokes tools
# ============================================================

class MCPClient:
    """A simplified MCP client that connects to servers and calls tools."""

    def __init__(self, name: str = "mcp-client"):
        self.name = name
        self.server: Optional[MCPServer] = None
        self.server_info: Dict[str, Any] = {}
        self.available_tools: List[Dict[str, Any]] = []
        self.connected = False
        self._log: List[str] = []

    def connect(self, server: MCPServer) -> bool:
        """Perform the initialization handshake with a server."""
        self._log.append(f"Connecting to server...")

        # Send initialize request
        init_request = MCPRequest(
            method="initialize",
            params={
                "protocolVersion": "2024-11-05",
                "clientInfo": {"name": self.name, "version": "1.0"},
                "capabilities": {}
            }
        )
        response = server.handle_request(init_request)

        if isinstance(response, MCPError):
            self._log.append(f"Connection failed: {response.message}")
            return False

        self.server = server
        self.server_info = response.result.get("serverInfo", {})
        self.connected = True
        self._log.append(f"Connected to: {self.server_info.get('name', 'unknown')}")

        # Discover available tools
        self._discover_tools()
        return True

    def _discover_tools(self) -> None:
        """Query the server for available tools."""
        if not self.connected:
            return

        request = MCPRequest(method="tools/list", params={})
        response = self.server.handle_request(request)

        if isinstance(response, MCPResponse):
            self.available_tools = response.result.get("tools", [])
            self._log.append(f"Discovered {len(self.available_tools)} tools")

    def list_available_tools(self) -> List[Dict[str, Any]]:
        """Return the list of tools discovered from the server."""
        return self.available_tools

    def call_tool(self, name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
        """Call a tool on the connected server."""
        if not self.connected:
            return {"error": "Not connected to any server"}

        request = MCPRequest(
            method="tools/call",
            params={"name": name, "arguments": arguments}
        )
        self._log.append(f"Calling tool: {name}({arguments})")

        response = self.server.handle_request(request)

        if isinstance(response, MCPError):
            self._log.append(f"Tool error: {response.message}")
            return {"error": response.message, "code": response.code}

        self._log.append(f"Tool result received")
        return {"result": response.result}

    def get_tool_names(self) -> List[str]:
        """Get just the names of available tools."""
        return [t["name"] for t in self.available_tools]

    def get_log(self) -> List[str]:
        return list(self._log)

# Test client-server connection
client = MCPClient("test-client")
success = client.connect(server)
print(f"Connected: {success}")
print(f"Server: {client.server_info}")
print(f"Available tools: {client.get_tool_names()}")

# Call a tool through the client
result = client.call_tool("greet", {"name": "MCP World"})
print(f"\nTool result: {result}")

# Show the communication log
print("\n--- Client Log ---")
for entry in client.get_log():
    print(f"  {entry}")

## 7. Example: Calculator MCP Server

Let's build a practical MCP server with multiple tools. A calculator server is a clean example because the tools have clear input schemas and deterministic outputs.

This demonstrates the core value proposition: **define tools once on the server, any client can discover and use them**.

In [ ]:
# ============================================================
# Calculator MCP Server — a complete example
# ============================================================

def build_calculator_server() -> MCPServer:
    """Build a calculator MCP server with 4 arithmetic tools."""
    server = MCPServer("calculator-server", version="1.0")

    # --- Tool handlers ---
    def add(a: float, b: float) -> str:
        return f"{a} + {b} = {a + b}"

    def subtract(a: float, b: float) -> str:
        return f"{a} - {b} = {a - b}"

    def multiply(a: float, b: float) -> str:
        return f"{a} × {b} = {a * b}"

    def divide(a: float, b: float) -> str:
        if b == 0:
            raise ValueError("Division by zero is undefined")
        return f"{a} ÷ {b} = {a / b}"

    # Common input schema for two-number operations
    two_numbers_schema = {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"}
        },
        "required": ["a", "b"]
    }

    server.register_tool("add", "Add two numbers.", two_numbers_schema, add)
    server.register_tool("subtract", "Subtract b from a.", two_numbers_schema, subtract)
    server.register_tool("multiply", "Multiply two numbers.", two_numbers_schema, multiply)
    server.register_tool("divide", "Divide a by b.", two_numbers_schema, divide)

    return server

calc_server = build_calculator_server()

# Connect a client and explore
calc_client = MCPClient("calc-user")
calc_client.connect(calc_server)

print("=== Calculator MCP Server ===")
print(f"Tools available: {calc_client.get_tool_names()}\n")

# Call each tool
for tool_name, args in [
    ("add", {"a": 15, "b": 27}),
    ("subtract", {"a": 100, "b": 37}),
    ("multiply", {"a": 6, "b": 7}),
    ("divide", {"a": 22, "b": 7}),
]:
    result = calc_client.call_tool(tool_name, args)
    content = result["result"]["content"][0]["text"]
    print(f"  {tool_name}: {content}")

# Test error handling — division by zero
print("\n--- Error handling ---")
result = calc_client.call_tool("divide", {"a": 10, "b": 0})
print(f"  divide(10, 0): {result}")

# Test error handling — unknown tool
result = calc_client.call_tool("sqrt", {"a": 16})
print(f"  sqrt(16): {result}")

## 8. Example: Text Processing MCP Server

A second server demonstrates that MCP servers are **domain-specific modules**. Each server encapsulates a set of related tools. Clients discover and use them without knowing implementation details.

In [ ]:
# ============================================================
# Text Processing MCP Server
# ============================================================

def build_text_server() -> MCPServer:
    """Build a text processing MCP server."""
    server = MCPServer("text-processing-server", version="1.0")

    def word_count(text: str) -> str:
        words = text.split()
        chars = len(text)
        sentences = text.count('.') + text.count('!') + text.count('?')
        return f"Words: {len(words)}, Characters: {chars}, Sentences: {max(sentences, 1)}"

    def reverse_text(text: str) -> str:
        return text[::-1]

    def extract_keywords(text: str, top_n: int = 5) -> str:
        words = text.lower().split()
        stop_words = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'it', 'in',
                      'on', 'at', 'to', 'for', 'of', 'and', 'or', 'but', 'with',
                      'that', 'this', 'from', 'by', 'as', 'be', 'has', 'have'}
        filtered = [w.strip('.,!?;:') for w in words if w.strip('.,!?;:') not in stop_words]
        freq = defaultdict(int)
        for w in filtered:
            if len(w) > 2:
                freq[w] += 1
        top = sorted(freq.items(), key=lambda x: x[1], reverse=True)[:top_n]
        return "Keywords: " + ", ".join(f"{w}({c})" for w, c in top)

    def text_stats(text: str) -> str:
        words = text.split()
        avg_word_len = sum(len(w) for w in words) / max(len(words), 1)
        unique_words = len(set(w.lower() for w in words))
        lexical_diversity = unique_words / max(len(words), 1)
        return (f"Avg word length: {avg_word_len:.1f} chars, "
                f"Unique words: {unique_words}/{len(words)}, "
                f"Lexical diversity: {lexical_diversity:.2%}")

    text_schema = {
        "type": "object",
        "properties": {"text": {"type": "string", "description": "Input text to process"}},
        "required": ["text"]
    }

    keyword_schema = {
        "type": "object",
        "properties": {
            "text": {"type": "string", "description": "Input text"},
            "top_n": {"type": "integer", "description": "Number of keywords to extract", "default": 5}
        },
        "required": ["text"]
    }

    server.register_tool("word_count", "Count words, characters, and sentences in text.", text_schema, word_count)
    server.register_tool("reverse_text", "Reverse the input text.", text_schema, reverse_text)
    server.register_tool("extract_keywords", "Extract top keywords from text.", keyword_schema, extract_keywords)
    server.register_tool("text_stats", "Compute text statistics (avg word length, lexical diversity).", text_schema, text_stats)

    return server

text_server = build_text_server()
text_client = MCPClient("text-user")
text_client.connect(text_server)

sample = "Model Context Protocol provides a standardized way for language models to interact with external tools and data sources efficiently."

print("=== Text Processing MCP Server ===")
print(f"Tools: {text_client.get_tool_names()}\n")

for tool_name in text_client.get_tool_names():
    args = {"text": sample}
    if tool_name == "extract_keywords":
        args["top_n"] = 3
    result = text_client.call_tool(tool_name, args)
    content = result["result"]["content"][0]["text"]
    print(f"  {tool_name}: {content}")

## 9. Building an MCP-Aware Agent

Now for the real payoff: an **agent that discovers tools from MCP servers at runtime**.

Unlike hardcoded tool agents where you write `if tool_name == "add": ...`, this agent:
1. Connects to one or more MCP servers
2. Queries each server for its available tools
3. Builds a tool catalog from the discovered schemas
4. Passes the catalog to the LLM for tool selection
5. Executes the chosen tool through the MCP protocol

The agent **never needs to know in advance** what tools exist.

In [ ]:
# ============================================================
# MCP-Aware Agent — discovers and uses tools dynamically
# ============================================================

class MCPAgent:
    """An agent that discovers tools from MCP servers and uses them."""

    def __init__(self, name: str = "mcp-agent"):
        self.name = name
        self.clients: Dict[str, MCPClient] = {}  # server_name → client
        self.tool_catalog: Dict[str, Dict[str, Any]] = {}  # tool_name → {server, schema}
        self._log: List[str] = []

    def connect_to_server(self, server: MCPServer) -> int:
        """Connect to an MCP server and discover its tools."""
        client = MCPClient(f"{self.name}-client-{server.name}")
        if not client.connect(server):
            self._log.append(f"Failed to connect to {server.name}")
            return 0

        self.clients[server.name] = client
        new_tools = 0

        for tool in client.list_available_tools():
            tool_name = tool["name"]
            self.tool_catalog[tool_name] = {
                "server": server.name,
                "schema": tool
            }
            new_tools += 1

        self._log.append(f"Connected to {server.name}: discovered {new_tools} tools")
        return new_tools

    def get_tool_descriptions(self) -> str:
        """Format all discovered tools for the LLM prompt."""
        if not self.tool_catalog:
            return "No tools available."

        lines = []
        for name, info in self.tool_catalog.items():
            schema = info["schema"]
            desc = schema.get("description", "No description")
            params = schema.get("inputSchema", {}).get("properties", {})
            param_str = ", ".join(
                f"{p}: {params[p].get('type', 'any')}"
                for p in params
            )
            lines.append(f"- {name}({param_str}): {desc}")

        return "\n".join(lines)

    def execute_tool(self, tool_name: str, arguments: Dict[str, Any]) -> str:
        """Execute a tool through the appropriate MCP server."""
        if tool_name not in self.tool_catalog:
            return f"Error: Tool '{tool_name}' not found in catalog."

        server_name = self.tool_catalog[tool_name]["server"]
        client = self.clients[server_name]

        result = client.call_tool(tool_name, arguments)

        if "error" in result:
            return f"Error: {result['error']}"

        content = result["result"]["content"]
        if content and len(content) > 0:
            return content[0].get("text", str(content))
        return str(result["result"])

    def run(self, user_query: str, max_steps: int = 5) -> str:
        """Run the agent with LLM-driven tool selection."""
        tool_descriptions = self.get_tool_descriptions()

        system_prompt = f"""You are a helpful assistant with access to these tools:

{tool_descriptions}

To use a tool, respond with EXACTLY this format:
TOOL: tool_name
ARGS: {{"param1": "value1", "param2": "value2"}}

To give a final answer, respond with:
ANSWER: your final answer here

Always use tools when they can help answer the question. Think step by step."""

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ]

        for step in range(max_steps):
            response = generate(messages, max_new_tokens=300, temperature=0.1, do_sample=True)
            self._log.append(f"Step {step + 1} LLM: {response[:100]}...")

            if "ANSWER:" in response:
                answer = response.split("ANSWER:", 1)[1].strip()
                self._log.append(f"Final answer: {answer[:100]}...")
                return answer

            if "TOOL:" in response and "ARGS:" in response:
                try:
                    tool_line = response.split("TOOL:", 1)[1].split("\n")[0].strip()
                    args_text = response.split("ARGS:", 1)[1].strip()
                    # Extract JSON from args
                    brace_start = args_text.index("{")
                    depth, end = 0, brace_start
                    for i, ch in enumerate(args_text[brace_start:], brace_start):
                        if ch == "{": depth += 1
                        elif ch == "}": depth -= 1
                        if depth == 0:
                            end = i + 1
                            break
                    args = json.loads(args_text[brace_start:end])

                    self._log.append(f"Calling: {tool_line}({args})")
                    result = self.execute_tool(tool_line, args)
                    self._log.append(f"Result: {result}")

                    messages.append({"role": "assistant", "content": response})
                    messages.append({"role": "user", "content": f"Tool result: {result}\nUse this to continue answering."})
                except (ValueError, json.JSONDecodeError) as e:
                    messages.append({"role": "assistant", "content": response})
                    messages.append({"role": "user", "content": f"Error parsing tool call: {e}. Try again with correct format."})
            else:
                return response

        return "Max steps reached without a final answer."

    def get_log(self) -> List[str]:
        return list(self._log)

print("MCPAgent class defined ✓")
print("The agent discovers tools at runtime via MCP protocol.")

## 10. Demo: Agent with Calculator Server

Let's connect our MCP agent to the calculator server and let the LLM decide which tools to use based on the user's question.

In [ ]:
# ============================================================
# Agent discovers and uses calculator tools via MCP
# ============================================================

agent = MCPAgent("calculator-agent")

# Connect to the calculator server — agent discovers tools automatically
num_tools = agent.connect_to_server(calc_server)
print(f"Agent discovered {num_tools} tools from calculator-server")
print(f"\nTool catalog:\n{agent.get_tool_descriptions()}\n")
print("=" * 60)

# Run the agent on a math question
query = "What is 42 multiplied by 17, and then add 100 to the result?"
print(f"Query: {query}\n")

answer = agent.run(query)
print(f"\nFinal answer: {answer}")

print("\n--- Agent Log ---")
for entry in agent.get_log():
    print(f"  {entry}

## 11. Multi-Server: Agent Connects to Multiple MCP Servers

The real power of MCP emerges when an agent connects to **multiple servers**. Each server is a domain-specific module, and the agent seamlessly combines tools from all of them.

```
                    ┌─────────────┐
                    │  MCP Agent  │
                    └──────┬──────┘
                           │ discovers tools from all servers
              ┌────────────┼────────────────┐
              ▼            ▼                ▼
       ┌────────────┐ ┌────────────┐ ┌────────────┐
       │ Calculator │ │   Text     │ │  (Future)  │
       │   Server   │ │   Server   │ │  DB Server │
       │ add, sub.. │ │ keywords.. │ │ query...   │
       └────────────┘ └────────────┘ └────────────┘
```

Adding a new server = the agent instantly gets new capabilities. No code changes needed.

In [ ]:
# ============================================================
# Multi-server agent — combines tools from multiple MCP servers
# ============================================================

multi_agent = MCPAgent("multi-server-agent")

# Connect to both servers
n1 = multi_agent.connect_to_server(calc_server)
n2 = multi_agent.connect_to_server(text_server)

print(f"=== Multi-Server Agent ===")
print(f"Connected to 2 servers, discovered {n1 + n2} total tools\n")
print(f"Full tool catalog:\n{multi_agent.get_tool_descriptions()}\n")

# The agent can now use tools from ANY connected server
print("=" * 60)
query = "Count the words in: 'The quick brown fox jumps over the lazy dog'"
print(f"Query: {query}\n")
answer = multi_agent.run(query)
print(f"\nAnswer: {answer}")

print("\n--- Agent Log ---")
for entry in multi_agent.get_log():
    print(f"  {entry}")

## 12. Dynamic Server Addition

One of MCP's strengths is that new servers can be added **at any time**. The agent re-discovers tools and immediately gains new capabilities — zero code changes, zero restarts.

Let's build a new unit conversion server and hot-plug it into our running agent.

In [ ]:
# ============================================================
# Dynamic server addition — hot-plug a new MCP server
# ============================================================

def build_conversion_server() -> MCPServer:
    """Build a unit conversion MCP server."""
    server = MCPServer("conversion-server", version="1.0")

    def celsius_to_fahrenheit(celsius: float) -> str:
        f = celsius * 9/5 + 32
        return f"{celsius}°C = {f:.1f}°F"

    def kg_to_pounds(kg: float) -> str:
        lbs = kg * 2.20462
        return f"{kg} kg = {lbs:.2f} lbs"

    def km_to_miles(km: float) -> str:
        miles = km * 0.621371
        return f"{km} km = {miles:.2f} miles"

    single_num = lambda desc: {
        "type": "object",
        "properties": {"celsius": {"type": "number", "description": desc}} if "temp" in desc else
                      {"kg": {"type": "number", "description": desc}} if "mass" in desc else
                      {"km": {"type": "number", "description": desc}},
        "required": ["celsius"] if "temp" in desc else ["kg"] if "mass" in desc else ["km"]
    }

    server.register_tool("celsius_to_fahrenheit", "Convert temperature from Celsius to Fahrenheit.",
        {"type": "object", "properties": {"celsius": {"type": "number", "description": "Temperature in Celsius"}}, "required": ["celsius"]},
        celsius_to_fahrenheit)
    server.register_tool("kg_to_pounds", "Convert mass from kilograms to pounds.",
        {"type": "object", "properties": {"kg": {"type": "number", "description": "Mass in kilograms"}}, "required": ["kg"]},
        kg_to_pounds)
    server.register_tool("km_to_miles", "Convert distance from kilometers to miles.",
        {"type": "object", "properties": {"km": {"type": "number", "description": "Distance in kilometers"}}, "required": ["km"]},
        km_to_miles)

    return server

# Before: show current tool count
print(f"Tools BEFORE adding new server: {len(multi_agent.tool_catalog)}")
print(f"  {list(multi_agent.tool_catalog.keys())}\n")

# Hot-plug new server
conv_server = build_conversion_server()
n3 = multi_agent.connect_to_server(conv_server)

# After: tools are immediately available
print(f"Tools AFTER adding conversion server: {len(multi_agent.tool_catalog)}")
print(f"  {list(multi_agent.tool_catalog.keys())}\n")
print(f"New tools added: {n3}")
print("\nNo code changes. No restarts. The agent simply discovered new tools.")

## 13. Comparison: Hardcoded Tools vs. MCP-Discovered Tools

To understand why MCP matters, let's compare the traditional approach (hardcoded tools) with the MCP approach (protocol-discovered tools) side by side.

In [ ]:
# ============================================================
# Hardcoded tools vs. MCP — side-by-side comparison
# ============================================================

# --- APPROACH 1: Hardcoded tools (the old way) ---
class HardcodedAgent:
    """An agent with tools baked into its code."""

    def __init__(self):
        self.tools = {
            "add": lambda a, b: a + b,
            "subtract": lambda a, b: a - b,
        }

    def call_tool(self, name, **kwargs):
        if name not in self.tools:
            return f"Unknown tool: {name}"
        return self.tools[name](**kwargs)

    def add_tool(self, name, func):
        """Adding a new tool requires modifying the agent code."""
        self.tools[name] = func
        # But the LLM prompt also needs updating...
        # And the tool description needs writing...
        # And error handling needs adding...

# --- APPROACH 2: MCP-discovered tools (the new way) ---
# (Already implemented above — MCPAgent)

# Compare the two approaches
print("=" * 60)
print("COMPARISON: Hardcoded vs. MCP-Discovered Tools")
print("=" * 60)

comparison = {
    "Adding a new tool": {
        "Hardcoded": "Modify agent source code, update prompt, redeploy",
        "MCP": "Register on server — agent discovers automatically"
    },
    "Tool portability": {
        "Hardcoded": "Tools locked to one agent implementation",
        "MCP": "Any MCP client can use any MCP server's tools"
    },
    "Discovery": {
        "Hardcoded": "Agent knows only what was coded in",
        "MCP": "Agent queries servers at runtime for capabilities"
    },
    "Schema validation": {
        "Hardcoded": "Manual — developer writes validation",
        "MCP": "Automatic — JSON Schema defines valid inputs"
    },
    "Multi-domain": {
        "Hardcoded": "All tools in one monolithic agent",
        "MCP": "Each domain is a separate server (modular)"
    },
    "Error handling": {
        "Hardcoded": "Custom per tool",
        "MCP": "Standardized error codes and messages"
    }
}

for aspect, approaches in comparison.items():
    print(f"\n{aspect}:")
    print(f"  Hardcoded: {approaches['Hardcoded']}")
    print(f"  MCP:       {approaches['MCP']}")

## 14. Capability Negotiation

When a client first connects to a server, they perform a **capability negotiation** handshake. This determines what features both sides support, preventing errors from unsupported operations.

This is directly inspired by real protocol design:
- **HTTP** uses content negotiation headers (`Accept`, `Content-Type`)
- **TLS** negotiates cipher suites during the handshake
- **MCP** negotiates which resource types (tools, prompts, resources) the server provides

In [ ]:
# ============================================================
# Capability negotiation — what each side supports
# ============================================================

class CapableServer(MCPServer):
    """Extended MCP server with richer capability negotiation."""

    def __init__(self, name: str, version: str = "1.0"):
        super().__init__(name, version)
        self.capabilities = {
            "tools": {"listChanged": True},
            "resources": {"subscribe": False, "listChanged": False},
            "prompts": {"listChanged": False},
            "logging": {}
        }
        self.supported_versions = ["2024-11-05"]

    def _handle_initialize(self, request: MCPRequest) -> MCPResponse:
        """Enhanced handshake with version and capability negotiation."""
        client_version = request.params.get("protocolVersion", "unknown")
        client_caps = request.params.get("capabilities", {})

        if client_version not in self.supported_versions:
            return MCPResponse(
                result={
                    "protocolVersion": self.supported_versions[0],
                    "serverInfo": {"name": self.name, "version": self.version},
                    "capabilities": self.capabilities,
                    "warning": f"Client requested {client_version}, server supports {self.supported_versions}"
                },
                id=request.id
            )

        return MCPResponse(
            result={
                "protocolVersion": client_version,
                "serverInfo": {"name": self.name, "version": self.version},
                "capabilities": self.capabilities,
                "negotiated": {
                    "tools": True,
                    "resources": "resources" in client_caps,
                    "prompts": "prompts" in client_caps
                }
            },
            id=request.id
        )

# Demonstrate capability negotiation
cap_server = CapableServer("capable-server")
cap_server.register_tool("ping", "Check if server is alive.",
    {"type": "object", "properties": {}, "required": []},
    lambda: "pong")

# Client with matching version
init_req = MCPRequest(method="initialize", params={
    "protocolVersion": "2024-11-05",
    "clientInfo": {"name": "demo-client"},
    "capabilities": {"tools": True}
})
resp = cap_server.handle_request(init_req)
print("Matching version handshake:")
print(json.dumps(resp.to_dict(), indent=2))

# Client with mismatched version
init_req2 = MCPRequest(method="initialize", params={
    "protocolVersion": "2025-01-01",
    "clientInfo": {"name": "future-client"},
    "capabilities": {}
})
resp2 = cap_server.handle_request(init_req2)
print("\nMismatched version handshake:")
print(json.dumps(resp2.to_dict(), indent=2))

## 15. Our Implementation vs. Full MCP Specification

We built a working MCP system, but the full specification includes much more. Here's what we covered and what we simplified:

### What We Built ✓
| Feature | Our Implementation |
|---------|-------------------|
| Tool registration | Full — name, description, JSON Schema |
| Tool discovery | Full — `tools/list` returns schemas |
| Tool execution | Full — `tools/call` with arguments |
| Error handling | Full — typed error codes, messages |
| JSON-RPC messages | Full — request/response/notification format |
| Capability negotiation | Basic — version and feature negotiation |
| Multi-server | Full — agent connects to multiple servers |

### What the Full Spec Adds
| Feature | Description |
|---------|-------------|
| **Transport layer** | Real stdio/HTTP+SSE communication (we used direct method calls) |
| **Resources** | Servers can expose data (files, DB records) not just tools |
| **Prompts** | Servers can provide prompt templates for common tasks |
| **Sampling** | Servers can request LLM completions from the client |
| **Progress reporting** | Long-running tools report progress back to client |
| **Cancellation** | Client can cancel in-flight tool calls |
| **Roots** | Client tells server which filesystem paths it has access to |
| **Subscriptions** | Client subscribes to resource change notifications |

### Key Takeaway

Our implementation captures the **core architecture** — the client-server model, JSON-RPC messaging, tool schemas, dynamic discovery, and multi-server composition. The full spec adds production concerns (transport, security, streaming) on top of these same fundamentals.

## 16. Validation: End-to-End MCP Flow

Let's run a comprehensive test that exercises the full protocol — initialization, discovery, tool calling, error handling, and multi-server composition.

In [ ]:
# ============================================================
# End-to-end validation of the MCP protocol implementation
# ============================================================

def run_mcp_tests():
    """Comprehensive test of MCP protocol components."""
    results = []

    # Test 1: Server registration and listing
    srv = MCPServer("test-srv")
    srv.register_tool("echo", "Echo input back.",
        {"type": "object", "properties": {"msg": {"type": "string"}}, "required": ["msg"]},
        lambda msg: msg)
    resp = srv.handle_request(MCPRequest(method="tools/list", params={}))
    assert len(resp.result["tools"]) == 1
    assert resp.result["tools"][0]["name"] == "echo"
    results.append(("Server tool registration", "PASS"))

    # Test 2: Tool execution success
    resp = srv.handle_request(MCPRequest(method="tools/call",
        params={"name": "echo", "arguments": {"msg": "hello"}}))
    assert "hello" in resp.result["content"][0]["text"]
    results.append(("Tool execution (success)", "PASS"))

    # Test 3: Tool not found error
    resp = srv.handle_request(MCPRequest(method="tools/call",
        params={"name": "nonexistent", "arguments": {}}))
    assert isinstance(resp, MCPError)
    assert resp.code == ErrorCode.TOOL_NOT_FOUND
    results.append(("Tool not found error", "PASS"))

    # Test 4: Invalid parameters error
    srv.register_tool("strict", "Needs x.",
        {"type": "object", "properties": {"x": {"type": "number"}}, "required": ["x"]},
        lambda x: x * 2)
    resp = srv.handle_request(MCPRequest(method="tools/call",
        params={"name": "strict", "arguments": {"wrong_param": 5}}))
    assert isinstance(resp, MCPError)
    assert resp.code == ErrorCode.INVALID_PARAMS
    results.append(("Invalid params error", "PASS"))

    # Test 5: Client connect and discover
    cli = MCPClient("test-cli")
    assert cli.connect(srv) == True
    assert cli.connected == True
    assert len(cli.available_tools) == 2
    results.append(("Client connect + discover", "PASS"))

    # Test 6: Client tool call
    result = cli.call_tool("echo", {"msg": "protocol test"})
    assert "protocol test" in result["result"]["content"][0]["text"]
    results.append(("Client tool call", "PASS"))

    # Test 7: Unknown method
    resp = srv.handle_request(MCPRequest(method="unknown/method", params={}))
    assert isinstance(resp, MCPError)
    assert resp.code == ErrorCode.METHOD_NOT_FOUND
    results.append(("Unknown method error", "PASS"))

    # Test 8: Multi-server agent
    srv2 = MCPServer("srv2")
    srv2.register_tool("ping", "Ping.", {"type": "object", "properties": {}}, lambda: "pong")
    agent = MCPAgent("test-agent")
    agent.connect_to_server(srv)
    agent.connect_to_server(srv2)
    assert len(agent.tool_catalog) == 3  # echo + strict + ping
    result = agent.execute_tool("ping", {})
    assert "pong" in result
    results.append(("Multi-server agent", "PASS"))

    # Test 9: JSON-RPC message format
    req = MCPRequest(method="test", params={"key": "val"})
    d = req.to_dict()
    assert d["jsonrpc"] == "2.0"
    assert d["method"] == "test"
    assert "id" in d
    results.append(("JSON-RPC format", "PASS"))

    # Test 10: Tool execution error handling
    srv.register_tool("fail", "Always fails.",
        {"type": "object", "properties": {}},
        lambda: 1/0)
    resp = srv.handle_request(MCPRequest(method="tools/call",
        params={"name": "fail", "arguments": {}}))
    assert isinstance(resp, MCPError)
    assert resp.code == ErrorCode.TOOL_EXECUTION_ERROR
    results.append(("Tool execution error", "PASS"))

    return results

tests = run_mcp_tests()
print("=== MCP Protocol Test Results ===\n")
for name, status in tests:
    icon = "✓" if status == "PASS" else "✗"
    print(f"  {icon} {name}: {status}")
print(f"\n{len(tests)}/{len(tests)} tests passed ✓")

## 17. Limitations and Future Directions

### Limitations of Our Implementation
1. **No real transport**: We call server methods directly instead of over stdio/HTTP. A production MCP implementation would serialize messages and send them over a transport layer.
2. **No async/streaming**: Real MCP supports streaming results for long-running tools and progress notifications. Our tools return synchronously.
3. **No authentication**: We don't handle auth. Real MCP servers would verify client identity.
4. **No schema validation**: We trust that the client sends valid arguments. A real implementation would validate against the JSON Schema before executing.
5. **No resource/prompt support**: We only implemented the tools capability. Full MCP also supports resources (data access) and prompts (template sharing).

### When MCP Shines
- **Tool marketplace**: Anyone can publish an MCP server, anyone can connect to it
- **Enterprise integration**: One protocol for all internal tools and APIs
- **Agent composability**: Agents gain new abilities by connecting to new servers
- **Separation of concerns**: Tool developers and agent developers work independently

### When MCP May Be Overkill
- **Single-use agents**: If your agent only ever uses 2-3 hardcoded tools
- **Latency-critical**: The protocol adds overhead (handshake, discovery, message serialization)
- **Simple scripts**: Not every Python script needs a protocol layer

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** extend the agent with a new tool. Document what changes and why.

**Exercise 2 — Build:** add error handling for a failure mode discussed in this notebook. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** trace through the agent loop manually and predict the output before running.

## 18. Summary — Key Takeaways

### What We Built
We implemented the **core of the Model Context Protocol** from scratch:

| Component | What It Does |
|-----------|-------------|
| **MCPRequest / MCPResponse / MCPError** | JSON-RPC 2.0 message format for structured communication |
| **ToolSchema** | JSON Schema-based tool definitions for dynamic discovery |
| **MCPServer** | Registers tools, handles protocol requests, returns results |
| **MCPClient** | Connects to servers, discovers tools, invokes them |
| **MCPAgent** | LLM-powered agent that discovers and uses tools via MCP |

### The Core Insight

> **MCP separates the protocol from the implementation**, just like HTTP separates web communication from web applications. This means any client can talk to any server, tools are discoverable at runtime, and the ecosystem can grow without coordination.

### Architecture Pattern

```
Agent (brain)  →  MCP Client (protocol)  →  MCP Server (tools)
     ↓                    ↓                       ↓
  "What tools          JSON-RPC               Register tools,
   should I use?"      messages                handle calls
```

### Next Steps
- **Notebook 30**: Agent-to-Agent (A2A) protocol — if MCP connects agents to tools, A2A connects agents to each other
- **Full MCP SDK**: Try the official `@modelcontextprotocol/sdk` (TypeScript) or `mcp` (Python) packages
- **Build a real server**: Wrap your favorite API as an MCP server and connect it to Claude Desktop

## References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the agents/ directory

---

## 🧭 Navigation

⬅️ **Previous:** [28 Error Handling And Resilience](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/28_error_handling_and_resilience.ipynb) | ➡️ **Next:** [30 A2A Protocol](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/30_a2a_protocol.ipynb)